# Data Quality Check
- Get a feel for the dataset (eg. number of rows, columns, values)
- Look for inconsistencies (eg. with date ranges, missing values)

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv ("ProjectDataset.csv")

In [4]:
df.shape

(4174500, 13)

**~4.2 M rows, with 13 columns.**

In [5]:
df.columns

Index(['state', 'playerid', 'wagerid', 'event_start', 'placed_date',
       'settled_date', 'sportname', 'bet_type', 'result', 'net_stake', 'ggr',
       'legresult', 'decimalodds'],
      dtype='object')

**What's the date range covered by the dataset?**

Below I went through all date columns (settled_date, placed_date, event_start) to ensure they roughly covered the same range.

**Note:** I expect event_start to cover a wider range, since patrons can bet on future events (eg. betting on the champion at the beginning of the season).

In [6]:

min_date = df["settled_date"].min()
max_date = df["settled_date"].max()
print (min_date,"to", max_date)

2021-03-28 to 2022-03-29


In [7]:
min_date = df["placed_date"].min()
max_date = df["placed_date"].max()
print (min_date,"to", max_date)

2021-03-28 to 2022-03-28


In [8]:
min_date = df["event_start"].min()
max_date = df["event_start"].max()
print (min_date,"to", max_date)

0001-12-30 00:00:00+00 to 2023-02-12 23:30:00+00


**Given betting activity from late March 2021 to late March 2022.**


**Issue 1:** Some records contain an invalid event_start value (0001-12-30 00:00:00+00), which indicates missing or placeholder event metadata.

**How many records are affected? How many contain a questionable event_start date?**

In [9]:
df[df["event_start"] < "2021-01-01"].shape[0]

453

**Potentially 453 records affected.**

**Are there any other missing values for these records that could affect my analysis?**

In [10]:
missing_event = df[df["event_start"] < "2021-01-01"]
missing_event.head()

,state,playerid,wagerid,event_start,placed_date,settled_date,sportname,bet_type,result,net_stake,ggr,legresult,decimalodds
7824,State1,22683957.12,4.051036e+06,0001-12-30 00:00:00+00,2021-04-08,2021-04-09,nba,parlay,won,21.020000,10.773333,won,1.08696
7831,State1,22683957.12,4.051074e+06,0001-12-30 00:00:00+00,2021-04-08,2021-04-09,nba,parlay,won,4.166666,2.421666,won,1.08696
7973,State1,22683957.12,4.076550e+06,0001-12-30 00:00:00+00,2021-04-09,2021-04-09,nba,parlay,won,5.000000,2.568000,won,1.13333
18641,State1,22177779.03,4.108515e+06,0001-12-30 00:00:00+00,2021-04-09,2021-04-09,nba,straight,won,10.000000,-9.090000,won,1.90909
30437,State1,22683957.12,4.098150e+06,0001-12-30 00:00:00+00,2021-04-09,2021-04-09,nba,parlay,won,33.333333,-38.743333,won,1.18182


In [11]:
# Confirming '0001-12-30 00:00:00+00' is the only value contained in these 453 rows
missing_event["event_start"].unique()

array(['0001-12-30 00:00:00+00'], dtype=object)

In [12]:
missing_event.isna().sum()

state           0
playerid        0
wagerid         0
event_start     0
placed_date     0
settled_date    0
sportname       0
bet_type        0
result          0
net_stake       0
ggr             0
legresult       0
decimalodds     0
dtype: int64

**There are no other missing values for the affected 453 records.**

**Hypothesis:** These records were missed when the event table was joined to the bets table to create the query, and thus given a default date.

Normally I would follow up with the team, but in this case I will proceed with the analysis noting that our **event_start column has ~400 rows that have a missing/placeholder value of (0001-12-30 00:00:00+00).**

**Are there any other missing values over the entire dataset?**

In [13]:
df.isna().sum().sort_values(ascending=False)

decimalodds     2472
playerid           0
wagerid            0
event_start        0
state              0
placed_date        0
settled_date       0
bet_type           0
sportname          0
result             0
net_stake          0
ggr                0
legresult          0
dtype: int64

In [14]:
missing = df[df["decimalodds"].isna()]
missing["bet_type"].value_counts()

bet_type
parlay    2472
Name: count, dtype: int64

In [15]:
missing["settled_date"].min(), missing["settled_date"].max()

('2021-03-29', '2021-06-30')

In [16]:
missing["sportname"].value_counts()

sportname
nba                   1823
college basketball     638
college football        11
Name: count, dtype: int64

In [17]:
missing.head()

,state,playerid,wagerid,event_start,placed_date,settled_date,sportname,bet_type,result,net_stake,ggr,legresult,decimalodds
936440,State1,2.227482e+07,4.895481e+07,2021-03-29 23:25:00+00,2021-03-29,2021-03-30,college basketball,parlay,won,5.00,-6.25,won,NaN
936441,State1,2.227482e+07,4.895481e+07,2021-03-30 02:02:00+00,2021-03-29,2021-03-30,college basketball,parlay,won,5.00,-6.25,won,NaN
936833,State1,2.886949e+07,4.911740e+07,2021-03-30 23:25:00+00,2021-03-30,2021-03-31,college basketball,parlay,lost,5.00,5.00,won,NaN
936834,State1,2.886949e+07,4.911740e+07,2021-03-31 02:07:00+00,2021-03-30,2021-03-31,college basketball,parlay,lost,5.00,5.00,lost,NaN
936850,State1,3.091458e+07,4.912196e+07,2021-03-31 02:07:00+00,2021-03-30,2021-03-31,college basketball,parlay,lost,4.84,4.84,lost,NaN


**Issue 2:** 2472 records were found where the decimal_odds column was Null. 

These records were all for parlays covering the NBA, college basketball & college football from March 29, 2021 to June 30, 2021.

**Hypothesis:** Since decimal odds vary per leg for a parlay, these values are stored in a table somewhere else, and for some reason weren't copied over when joining the legs table to the bets table to make this CSV.

**I can safely keep these rows, since I have all other information regarding these bets (above we confirm these records are not missing other values).**

Normally I would follow up with the team, but in this case I proceed with analysis noting **odds may not always be present for parlay legs in the dataset.**

**Now I will modify column names, and take a further look into the values of the categorical columns.**

In [18]:
# renaming columns to what I'm used to working with at theScore
rename_map = {
    "playerid": "player_id",
    "wagerid": "wager_id",
    "sportname": "sport",
    "result": "bet_result",
    "legresult": "leg_result",
    "decimalodds": "decimal_odds",
    "net_stake": "handle",
    "decimal_odds": "odds"
}

df = df.rename(columns=rename_map)
df.columns

Index(['state', 'player_id', 'wager_id', 'event_start', 'placed_date',
       'settled_date', 'sport', 'bet_type', 'bet_result', 'handle', 'ggr',
       'leg_result', 'decimal_odds'],
      dtype='object')

**I want to understand the unique values contained in the main categorical columns.**

In [19]:
categorical_cols = ["state", "sport", "bet_type", "bet_result", "leg_result"]

for col in categorical_cols:
    print(f"\n--- {col.upper()} ---")
    print("Distinct values:", df[col].nunique())
    print(df[col].unique())


--- STATE ---
Distinct values: 3
['State1' 'State3' 'State2']

--- SPORT ---
Distinct values: 7
['nhl' 'nba' 'mlb' 'champions league' 'nfl' 'college football'
 'college basketball']

--- BET_TYPE ---
Distinct values: 2
['straight' 'parlay']

--- BET_RESULT ---
Distinct values: 2
['won' 'lost']

--- LEG_RESULT ---
Distinct values: 5
['won' 'lost' 'void' 'open' 'unknown']


**Now I inspect records with leg_result in (void, open, unknown), as I want to see if they match my expectations.**

In [20]:
void_df = df[df["leg_result"] == "void"]

void_count = void_df.shape[0]

void_count


23669

In [21]:
void_straight_nonzero = df[
    (df["leg_result"] == "void") &
    (df["bet_type"] == "straight") &
    (df["ggr"] != 0)
]

void_straight_nonzero.shape[0]

153

In [22]:
void_straight_nonzero.head()

,state,player_id,wager_id,event_start,placed_date,settled_date,sport,bet_type,bet_result,handle,ggr,leg_result,decimal_odds
2134,State1,3.131491e+07,3.236234e+06,2021-05-11 23:45:00+00,2021-05-11,2021-05-11,nba,straight,won,10.00,1.57,void,1.40816
9289,State1,2.801471e+07,2.454171e+06,2021-05-04 23:46:00+00,2021-05-04,2021-05-04,mlb,straight,won,8.50,2.80,void,5.30000
10843,State3,2.496263e+07,3.404011e+06,2021-09-10 00:27:00+00,2021-09-09,2021-09-09,nfl,straight,won,10.00,-0.39,void,1.60606
17245,State1,3.007023e+07,2.745815e+06,2021-05-07 02:15:00+00,2021-05-06,2021-05-06,nba,straight,won,25.00,-0.72,void,1.90909
44019,State1,2.885174e+07,3.562072e+06,2021-05-15 17:15:00+00,2021-05-15,2021-05-15,nba,straight,won,81.52,34.11,void,2.22000


**Issue 3:** I expected **straight bets with a leg_result as void** to always have 0 GGR (full refund).

**I'm going to proceed with leg_result being void providing additonal context for a bet but understanding it cannot be used alone to determine wager outcome/refund status (I would need further clarification from the team).**

In [23]:
open_df = df[df["leg_result"] == "open"]
open_count = open_df.shape[0]
open_count


2614

In [24]:
open_df["bet_result"].value_counts()


bet_result
won     2158
lost     456
Name: count, dtype: int64

In [25]:
open_df[open_df["ggr"] != 0].shape[0]



975

In [26]:
open_df.head()

,state,player_id,wager_id,event_start,placed_date,settled_date,sport,bet_type,bet_result,handle,ggr,leg_result,decimal_odds
2273,State3,3.258380e+07,3.313868e+06,2022-07-20 23:00:00+00,2021-09-09,2021-10-19,nba,parlay,won,5.0,3.835,open,5.00000
6783,State3,3.162506e+07,3.766224e+06,2021-12-01 00:00:00+00,2021-09-12,2021-09-13,college basketball,straight,won,10.0,0.000,open,31.00000
18095,State1,4.090628e+06,1.869826e+06,2021-04-30 00:11:00+00,2021-04-28,2021-04-28,mlb,straight,won,45.0,0.000,open,1.60976
32895,State3,3.525885e+07,4.807805e+06,2021-10-12 23:00:00+00,2021-09-19,2021-09-22,nhl,straight,won,5.0,0.000,open,21.00000
32896,State3,3.525885e+07,4.807818e+06,2021-10-12 23:00:00+00,2021-09-19,2021-09-22,nhl,straight,won,5.0,0.000,open,151.00000


**Isssue 4a:** 2614 records with **leg_result as open** have a valid bet_result, settled_date and sometimes non-zero GGR (would expect open bets to have 0 GGR).

**Hypothesis:** bet_legs failing to update after the join between the legs table and bets table (maybe the legs table wasn't refreshed).

**Once again I would need to follow up with the team to confirm, but for now we can assume that leg_result = 'open' is not reliable.**

In [27]:
unknown_df = df[df["leg_result"] == "unknown"]
unknown_count = unknown_df.shape[0]
unknown_count


41

In [28]:
unknown_df[unknown_df["ggr"] != 0].shape[0]


40

In [29]:
unknown_df[unknown_df["ggr"] != 0].head()


,state,player_id,wager_id,event_start,placed_date,settled_date,sport,bet_type,bet_result,handle,ggr,leg_result,decimal_odds
758634,State1,1.480849e+07,3.728608e+07,2022-02-15 00:41:00+00,2022-02-14,2022-02-14,nba,straight,lost,26.200000,26.200000,unknown,1.86207
1793352,State1,1.001368e+07,3.725825e+07,2022-02-15 02:13:00+00,2022-02-14,2022-02-14,nba,straight,lost,10.000000,10.000000,unknown,1.86207
1793500,State1,1.281213e+06,3.729581e+07,2022-02-15 01:11:00+00,2022-02-14,2022-02-14,nba,parlay,lost,8.333333,8.333333,unknown,2.96000
1793501,State1,1.281213e+06,3.729581e+07,2022-02-15 01:11:00+00,2022-02-14,2022-02-14,nba,parlay,lost,8.333333,8.333333,unknown,2.02000
1793502,State1,1.281213e+06,3.729581e+07,2022-02-15 01:11:00+00,2022-02-14,2022-02-14,nba,parlay,lost,8.333333,8.333333,unknown,1.86207


**Issue 4b:** 41 records with **leg_result as unknown** have a valid bet_result, settled_date and almost always non-zero GGR (would expect unknown bets to have 0 GGR).

I would need to follow up with the team to confirm, but **for now we can assume that leg_result = 'unknown' is not reliable.**

**Overall, I found that the leg-level metadata is not always synchronized with each bet. Therefore, while leg_result can be used for supplemental context, it should not be treated as a reliable indicator of financial outcome or refund status.**